# 05 — Transfer learning: pretrained ResNet18, RGB vs 13 spectral bands

**Objective:** Fine-tune a pretrained ResNet18 for the same 10 EuroSAT land-cover labels, using exactly the **spatial train/validation/test split and train-only band statistics from notebook 02**. Run 3 seeds for RGB and for all 13 bands. The 13-band convolution is initialized with the pretrained R/G/B filters at their corresponding channels; weights for the 10 additional channels start at zero and learn during fine-tuning. This gives RGB and 13-band comparable initial *visible-band* feature extraction; non-RGB bands are not pretrained.

**Requires:** `results/split.csv`, `results/band_stats.csv`, `data/eurosat_cache.npy` (or raw EuroSAT_MS GeoTIFFs), PyTorch, and a compatible `torchvision`. On first use **ImageNet-1K pretrained weights must download**; do not silently substitute random weights. Run inside the cloned repository, not a downloaded standalone notebook.

**Outputs:** `models/resnet_<input>_seed<n>.pt` (local; never commit), `results/predictions/resnet_<input>_seed<n>.csv`, `results/resnet_results.csv`, `results/resnet_history.csv`, figures in `results/figures/`.

**Resource note:** This trains **six ResNet runs**, much heavier than notebook 06. Do **not** train concurrently with notebook 04. Default input resolution is 128×128 to reduce memory/time; the original proposal envisaged 224×224. If you use 128, report the deviation; `INPUT_SIZE = 224` is available for the full proposed setting. This is *selective* fine-tuning (last residual stage and classifier), not all-layer full fine-tuning. ImageNet pretrained features and train-only EuroSAT normalization are a known domain shift. Save seed and configuration for honest comparison. Always select the checkpoint on validation loss; no test-driven hyperparameter tuning.

## 1. Setup and shared data

Identical label order, row order, preprocessing, held-out test and buffers as notebook 02. The project root is detected automatically whether the Jupyter working directory is the repository root or `notebooks/`. The cache is memory-mapped, not read wholesale into memory.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

def find_project_root():
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / 'src' / 'piece_of_crop' / 'data.py').is_file() and (candidate / 'notebooks').is_dir():
            return candidate
    raise FileNotFoundError('Open the cloned piece_of_crop repository in VS Code (File > Open Folder).')

project_root = find_project_root()
src_dir = str(project_root / 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
results_dir = project_root / 'results'
predictions_dir = results_dir / 'predictions'
figures_dir = results_dir / 'figures'
models_dir = project_root / 'models'
for directory in (results_dir, predictions_dir, figures_dir, models_dir):
    directory.mkdir(parents=True, exist_ok=True)
print('Project root:', project_root)

import copy
import random
import torch
from torch import nn
from torch.nn import functional as F
from sklearn.metrics import accuracy_score, f1_score, classification_report
try:
    from torchvision.models import resnet18, ResNet18_Weights
except Exception as exc:
    raise ImportError('A working torchvision installation matching PyTorch is required. '
                      'Check environment.yml and the PyTorch/torchvision installation.') from exc
from piece_of_crop import get_cache, make_loaders, read_split, CLASS_NAMES

SEEDS = (0, 1, 2)
BATCH_SIZE = 32
MAX_EPOCHS = 6
PATIENCE = 2
INPUT_SIZE = 128  # change to 224 for the originally proposed resolution; report choice
LR_BACKBONE = 1e-4
LR_HEAD = 1e-3
WEIGHT_DECAY = 1e-4
SKIP_COMPLETED = True  # resume completed configurations; do not skip incomplete ones

split_path = results_dir / 'split.csv'
stats_path = results_dir / 'band_stats.csv'
if not split_path.is_file() or not stats_path.is_file():
    raise FileNotFoundError('Run notebook 02 first to create split.csv and band_stats.csv.')
split = read_split(split_path)
band_stats = pd.read_csv(stats_path)
cache = get_cache(split, project_root / 'data' / 'EuroSAT_MS', project_root / 'data' / 'eurosat_cache.npy')
label_ids = np.arange(len(CLASS_NAMES), dtype=int)
assert np.array_equal(np.sort(split['label'].unique()), label_ids), 'Unexpected class IDs'
assert cache.shape == (len(split), 13, 64, 64), 'Cache/split mismatch'

if torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print('Device:', device, 'Split:', split['split'].value_counts().to_dict())
print('Config:', 'seeds', SEEDS, 'batch', BATCH_SIZE, 'input', INPUT_SIZE, 'max epochs', MAX_EPOCHS)

## 2. Adapt ResNet18 to 3 vs 13 bands

ImageNet conv1 weights are in R/G/B order. The project stores Sentinel-2 bands as `B01, B02, B03, B04, ..., B8A`; RGB loader returns **B04, B03, B02** (indices 3, 2, 1). For the 13-band model put those three pretrained filters in the corresponding positions and initialize extra-channel weights to zero. Keep the same pretrained backbone and replace the 1000-class head with a 10-class head. This is a documented initialization choice, not equivalent to ImageNet pretraining on all 13 channels.

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def build_resnet(input_type, pretrained=True):
    if input_type not in ('rgb', '13band'):
        raise ValueError('input_type must be rgb or 13band')
    weights = ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    net = resnet18(weights=weights)
    if input_type == '13band':
        old = net.conv1
        new = nn.Conv2d(13, old.out_channels, kernel_size=old.kernel_size,
                        stride=old.stride, padding=old.padding, bias=False)
        with torch.no_grad():
            new.weight.zero_()
            new.weight[:, 3] = old.weight[:, 0]  # B04 = red
            new.weight[:, 2] = old.weight[:, 1]  # B03 = green
            new.weight[:, 1] = old.weight[:, 2]  # B02 = blue
        net.conv1 = new
    net.fc = nn.Linear(net.fc.in_features, len(CLASS_NAMES))
    # Fine-tune last residual stage + classification head, keep earlier backbone frozen.
    for parameter in net.parameters():
        parameter.requires_grad_(False)
    for parameter in net.layer4.parameters():
        parameter.requires_grad_(True)
    for parameter in net.fc.parameters():
        parameter.requires_grad_(True)
    # For 13 bands the new first layer must learn to use the other 10 channels.
    if input_type == '13band':
        for parameter in net.conv1.parameters():
            parameter.requires_grad_(True)
    return net

# Structural checks require no network/download.
for input_type, channels in [('rgb', 3), ('13band', 13)]:
    model_check = build_resnet(input_type, pretrained=False)
    assert model_check.conv1.in_channels == channels
    with torch.no_grad():
        assert model_check(torch.zeros(2, channels, 64, 64)).shape == (2, 10)
    del model_check
print('ResNet18 3- and 13-channel shape checks passed (without pretrained download).')

## 3. Training helpers and evaluation

Resize identically for both inputs immediately before the network. Earlier convolutional layers stay frozen. BatchNorm statistics in frozen layers are held fixed in training by setting those modules to evaluation mode. Validation/test loaders have fixed order. Save model checkpoints from the **lowest validation loss**; compute macro-F1 on the held-out test only after checkpoint selection.

In [ ]:
def resize_for_model(images):
    return F.interpolate(images, size=(INPUT_SIZE, INPUT_SIZE),
                         mode='bilinear', align_corners=False)


def train_epoch(net, loader, optimizer, loss_fn):
    net.train()
    # Keep BatchNorm running statistics frozen in the pretrained backbone.
    for module in net.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()
    total_loss, n_seen, n_correct = 0., 0, 0
    for images, labels in loader:
        images = resize_for_model(images.to(device=device, dtype=torch.float32))
        labels = labels.to(device=device, dtype=torch.long)
        optimizer.zero_grad(set_to_none=True)
        logits = net(images)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        n = len(labels)
        total_loss += loss.item() * n
        n_seen += n
        n_correct += (logits.argmax(1) == labels).sum().item()
    return {'loss':total_loss/n_seen, 'accuracy':n_correct/n_seen}


@torch.inference_mode()
def evaluate_resnet(net, loader, loss_fn):
    net.eval()
    total_loss, n_seen = 0., 0
    true_list, proba_list = [], []
    for images, labels in loader:
        images = resize_for_model(images.to(device=device, dtype=torch.float32))
        labels = labels.to(device=device, dtype=torch.long)
        logits = net(images)
        total_loss += loss_fn(logits, labels).item() * len(labels)
        n_seen += len(labels)
        true_list.append(labels.cpu().numpy())
        proba_list.append(torch.softmax(logits, dim=1).cpu().numpy())
    y_true = np.concatenate(true_list)
    probabilities = np.concatenate(proba_list)
    y_pred = probabilities.argmax(axis=1)
    return dict(loss=total_loss/n_seen,
                accuracy=accuracy_score(y_true,y_pred),
                macro_f1=f1_score(y_true,y_pred,labels=label_ids,average='macro',zero_division=0),
                y_true=y_true,y_pred=y_pred,proba=probabilities)


def fit_resnet(net, loaders, seed, input_type):
    param_groups = [
        {'params':net.layer4.parameters(), 'lr':LR_BACKBONE},
        {'params':net.fc.parameters(), 'lr':LR_HEAD},
    ]
    if input_type == '13band':
        param_groups.append({'params':net.conv1.parameters(), 'lr':LR_BACKBONE})
    opt = torch.optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.CrossEntropyLoss()
    best_loss, best_state, stale = float('inf'), None, 0
    history = []
    for epoch in range(1, MAX_EPOCHS + 1):
        train = train_epoch(net, loaders['train'], opt, loss_fn)
        val = evaluate_resnet(net, loaders['val'], loss_fn)
        history.append({'model':'ResNet18','input':input_type,'seed':seed,'epoch':epoch,
                        'train_loss':train['loss'],'train_accuracy':train['accuracy'],
                        'val_loss':val['loss'],'val_accuracy':val['accuracy'],
                        'val_macro_f1':val['macro_f1']})
        print(f'{input_type} seed={seed} ep={epoch:02d} train={train["loss"]:.4f} '
              f'val={val["loss"]:.4f} val_acc={val["accuracy"]:.3f} '
              f'val_F1={val["macro_f1"]:.3f}', flush=True)
        if val['loss'] < best_loss - 1e-4:
            best_loss = val['loss']
            best_state = copy.deepcopy(net.state_dict())
            stale = 0
        else:
            stale += 1
            if stale >= PATIENCE:
                print('Early stopping (validation loss).')
                break
    if best_state is None:
        raise RuntimeError('No validation checkpoint could be selected.')
    net.load_state_dict(best_state)
    return history, loss_fn

## 4. Run the six configurations (3 seeds × 2 spectral inputs)

**Compute-intensive**: only run this cell when notebook 04 is finished and the Mac has enough free memory. Each completed seed is saved immediately, so an interrupted run can resume. The very first pretrained model creation downloads ImageNet weights. Do not use `torchvision` models with `weights=None` for the actual experiment. Keep `SKIP_COMPLETED=True` unless you deliberately want to rerun a seed with new settings; if settings change, remove/archive old results and checkpoints first to prevent mixing experiments.

In [ ]:
history_path = results_dir / 'resnet_history.csv'
metrics_path = results_dir / 'resnet_results.csv'

if SKIP_COMPLETED:
    saved_history = pd.read_csv(history_path).to_dict('records') if history_path.exists() else []
    saved_metrics = pd.read_csv(metrics_path).to_dict('records') if metrics_path.exists() else []
else:
    saved_history, saved_metrics = [], []

config_tag = f'resnet18_size{INPUT_SIZE}_ep{MAX_EPOCHS}_bs{BATCH_SIZE}_lr{LR_BACKBONE}_{LR_HEAD}'
for input_type in ('rgb', '13band'):
    for seed in SEEDS:
        weight_path = models_dir / f'resnet_{input_type}_seed{seed}.pt'
        prediction_path = predictions_dir / f'resnet_{input_type}_seed{seed}.csv'
        prior_rows = [r for r in saved_metrics if r.get('input') == input_type and int(r.get('seed', -1)) == seed]
        completed = (weight_path.exists() and prediction_path.exists() and
                     {r.get('split') for r in prior_rows} == {'val', 'test'} and
                     all(r.get('config') == config_tag for r in prior_rows))
        if completed and SKIP_COMPLETED:
            print(f'Skipping completed {input_type}, seed {seed}')
            continue

        print(f'\n=== ResNet18 {input_type} / seed {seed} ===', flush=True)
        saved_history = [r for r in saved_history if not (r.get('input')==input_type and int(r.get('seed',-1))==seed)]
        saved_metrics = [r for r in saved_metrics if not (r.get('input')==input_type and int(r.get('seed',-1))==seed)]
        set_seed(seed)
        loaders = make_loaders(split, cache, band_stats, input_type,
                               batch_size=BATCH_SIZE, seed=seed)
        try:
            net = build_resnet(input_type, pretrained=True).to(device)
        except Exception as exc:
            raise RuntimeError('Pretrained ResNet18 weights could not be loaded/downloaded. '
                               'Check internet access and torchvision installation; '
                               'do not continue with randomly initialized weights.') from exc
        history, loss_fn = fit_resnet(net, loaders, seed, input_type)

        for part in ('val', 'test'):
            outcome = evaluate_resnet(net, loaders[part], loss_fn)
            saved_metrics.append({'model':'ResNet18','input':input_type,'seed':seed,
                                  'split':part,'accuracy':outcome['accuracy'],
                                  'macro_f1':outcome['macro_f1'],'loss':outcome['loss'],
                                  'n_samples':len(outcome['y_true']), 'config': config_tag})
            if part == 'test':
                expected = split.loc[split['split'].eq('test'), ['file', 'label']].reset_index(drop=True)
                if not np.array_equal(outcome['y_true'], expected['label'].to_numpy(dtype=int)):
                    raise ValueError('Test DataLoader order/labels differ from split.csv.')
                out = pd.DataFrame({'file':expected['file'],'y_true':outcome['y_true'],
                                    'y_pred':outcome['y_pred'],
                                    'confidence':outcome['proba'].max(axis=1)})
                for cls in label_ids:
                    out[f'p_{cls}'] = outcome['proba'][:,cls]
                out.to_csv(prediction_path, index=False)

        torch.save({'state_dict':{k:v.detach().cpu() for k,v in net.state_dict().items()},
                    'model':'ResNet18','input':input_type,'seed':seed,
                    'num_classes':len(CLASS_NAMES),'input_size':INPUT_SIZE,
                    'config':config_tag}, weight_path)
        saved_history.extend([{**r, 'config':config_tag} for r in history])
        pd.DataFrame(saved_history).to_csv(history_path, index=False)
        pd.DataFrame(saved_metrics).to_csv(metrics_path, index=False)
        print('Saved:', weight_path.name, prediction_path.name, flush=True)
        del net, loaders
        if device.type == 'mps' and hasattr(torch, 'mps'):
            torch.mps.empty_cache()

resnet_history = pd.DataFrame(saved_history)
resnet_results = pd.DataFrame(saved_metrics)
print('Configurations completed:', len(resnet_results.query("split == 'test'")), '/ 6')
display(resnet_results.sort_values(['input','seed','split']))

## 5. Plot validation learning curves and model comparison

All shown values come from completed training runs. An incomplete run does not produce a fabricated score.

In [ ]:
import matplotlib.pyplot as plt

if resnet_history.empty or resnet_results.empty:
    raise RuntimeError('No ResNet training results yet; run the training cell first.')
for input_type in ('rgb', '13band'):
    group = resnet_history.query('input == @input_type')
    if group.empty:
        continue
    fig, axes = plt.subplots(1, 2, figsize=(11,4))
    for seed, run in group.groupby('seed'):
        axes[0].plot(run['epoch'], run['train_loss'], ls='--',label=f'train {seed}')
        axes[0].plot(run['epoch'], run['val_loss'],label=f'val {seed}')
        axes[1].plot(run['epoch'], run['val_accuracy'],label=f'val {seed}')
    axes[0].set(title=f'ResNet18 {input_type}: loss', xlabel='Epoch', ylabel='Cross-entropy')
    axes[1].set(title=f'ResNet18 {input_type}: accuracy',xlabel='Epoch',ylabel='Accuracy',ylim=(0,1))
    for ax in axes:
        ax.grid(alpha=.25); ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(figures_dir / f'resnet_{input_type}_training.png', dpi=150, bbox_inches='tight')
    plt.show()

print('Test mean and std across completed seeds:')
display(resnet_results.query("split == 'test'").groupby('input')
        [['accuracy','macro_f1']].agg(['mean','std']).round(4))
for input_type in ('rgb','13band'):
    path = predictions_dir / f'resnet_{input_type}_seed0.csv'
    if path.is_file():
        pred = pd.read_csv(path)
        print(f'\nPer-class test report: {input_type}, seed 0')
        print(classification_report(pred['y_true'],pred['y_pred'], labels=label_ids,
                                    target_names=list(CLASS_NAMES),zero_division=0))

## 6. Methodological limitations for the report

- The comparison is **RGB vs 13-band** with identical splits and train-derived normalization. No test set was used to tune training settings.
- Pretraining on ImageNet RGB photos advantages the RGB pathway. The 13-band conv1 has only three pretrained visible-channel filters; its extra bands are learned from EuroSAT. **This is not 13-band pretraining.**
- The default 128×128 resolution differs from the originally proposed 224×224; document the actual resolution and selective layer4+head fine-tuning.
- These data contain land-cover labels, **not crop loss, yield or insurance-claim values**. Predictions are decision support for human review, not automatic insurance decisions.
- A spatial split reduces but does not eliminate region/scene leakage, domain shift and annotation ambiguity. Three seeds indicate sensitivity to initialization but are not a full statistical guarantee.

**External implementation reference:** [Torchvision ResNet18 documentation](https://docs.pytorch.org/vision/2.0/models/generated/torchvision.models.resnet18.html) describes `resnet18` and `ResNet18_Weights.IMAGENET1K_V1`. Actual class, split and input processing are defined by the team notebooks, not by ImageNet default transforms.